# SpatialData Conversion - Comprehensive Feature Test

This notebook exercises the full `insitupy.spatialdata` conversion surface in one place, as a
manual smoke test for recent work (see `.log/log.md` around 2026-07-11 to 2026-07-14):

- the InSituPy-native dialect round trip: `convert_to_spatialdata` / `convert_from_spatialdata` /
  `read_spatialdata` (single sample, multi-sample `InSituExperiment`, and lazy experiment views)
- the **spatial units** modality (e.g. Visium spots) on both the export and dialect-import sides
- **`convert_from_foreign_spatialdata`** - the keyed-dict importer for non-InSituPy /
  labels-native SpatialData stores (`spatialdata_io.xenium()`, `spatialdata_io.visium()`, or any
  other store following the standard SpatialData table-annotation contract), including the new
  multi-layer cell/unit import capability and the spec validation it added
- the concatenated-table (`TABLES.<layer>`) export/import round trip added for
  `InSituExperiment.build_table()`

It builds on and supersedes the narrower `tutorial_convert_to_spatialdata.ipynb` and
`tutorial_convert_from_spatialdata.ipynb` in this same folder (those still work for the basic
single-purpose walkthroughs; this notebook is the "test everything" companion). It also reuses
the Visium download helper from `../03_data_import/InSituPy_visium_xenium_alignment.ipynb`.

**Prerequisites:** the demo Xenium project from `../01_demo_analysis/00_InSituPy_demo_datasets.ipynb`
should already exist at `CACHE / "out/demo_insitupy_project"` (used in Parts 1-2 and 5).

## 0. Setup

In [1]:
# Enable autoreload for development
%load_ext autoreload
%autoreload 2

### Make sure `SpatialData` is installed

If it is not installed yet, install it with:
```bash
pip install spatialdata[extra] spatialdata-io
```
InSituPy pins `spatialdata>=0.8.0,<0.9.0` (see `pyproject.toml`) - everything in this notebook is
developed and tested against `spatialdata==0.8.0`, paired with a pinned `zarr>=3.2.1,<4.0.0` so
exported stores are deterministically zarr v3.

In [2]:
import numpy as np

from insitupy import CACHE, InSituData, InSituExperiment
from insitupy.spatialdata import (
    convert_from_foreign_spatialdata,
    convert_from_spatialdata,
    convert_table_from_spatialdata,
    convert_to_spatialdata,
    read_spatialdata,
)

## Part 1 - InSituPy-native dialect round trip (single sample)

### 1.1 Load an `InSituData` object

In [3]:
xd = InSituData.read(CACHE / "out/demo_insitupy_project")
xd.load_all()
xd

C:\Users\ge37voy\Github\insitupy\insitupy\containers\io.py:308: UserWarning: Saved nucleus_to_cell_map uses the legacy position-based format and is inconsistent with the cell table (likely filtered by an older InSituPy that did not maintain it). Dropping it; nuclei will be treated as 1:1 with cells. Re-read from raw data to restore multinucleated-cell mapping.
  boundaries = _read_boundaries_from_celldata_zarr(bound_path)
C:\Users\ge37voy\Github\insitupy\insitupy\containers\io.py:308: UserWarning: Saved nucleus_to_cell_map uses the legacy position-based format and is inconsistent with the cell table (likely filtered by an older InSituPy that did not maintain it). Dropping it; nuclei will be treated as 1:1 with cells. Re-read from raw data to restore multinucleated-cell mapping.
  boundaries = _read_boundaries_from_celldata_zarr(bound_path)


InSituData
Method:		Xenium
Slide ID:	0001879
Sample ID:	Replicate 1
UID:		None
Path:		C:\Users\ge37voy\.cache\InSituPy\out\demo_insitupy_project

    ➤ images
       'CD20':     (25778, 35416)
       'HE':       (25778, 35416, 3)
       'HER2':     (25778, 35416)
       'nuclei':   (25778, 35416)
    ➤ cells
       MultiCellData with main layer 'main'
           table
               AnnData object with n_obs × n_vars = 157600 × 297
               obs: 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_genes_by_counts', 'n_genes', 'leiden'
               var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
               uns: 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'umap'
               obsm: 'X_pca', 'X_umap', 'spatial'
               varm: 'PCs'
               layers: 'counts', 'norm_counts'
               obsp: 'connectivit

In [4]:
xd.show()

2026-07-21 13:19:55 | [INFO] No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


2026-07-21 13:20:08 | [INFO] Extracting unique gene names from Dask DataFrame...
2026-07-21 13:20:11 | [INFO] Found 541 unique genes


C:\Users\ge37voy\Github\insitupy\insitupy\_core\_napari.py:615: FutureWarning: Setting unit on the ScaleBar model is deprecated. Units will instead be computed from the layers in the layerlist. To silence this warning, leave scale_bar unit as `None`, and use `Layer.units` to set units for each layer. Starting in v0.8.0, setting ScaleBar.unit will no longer have an effect. Starting from v0.9.0, it will be removed and raise an exception.
  current_viewer.scale_bar.unit = unit


### 1.2 Export to SpatialData with `convert_to_spatialdata`

Integrates every modality (images, cells, units, transcripts, annotations, regions) into one
`SpatialData` object under InSituPy's own naming dialect.

In [5]:
sdata = convert_to_spatialdata(xd)
sdata

2026-07-21 13:20:21 | [INFO] No case-insensitive conflicts found.


SpatialData object
├── Images
│     ├── 'IMAGES.CD20': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213), (1, 805, 1106)
│     ├── 'IMAGES.HE': DataTree[cyx] (3, 25778, 35416), (3, 12889, 17708), (3, 6444, 8854), (3, 3222, 4427), (3, 1611, 2213), (3, 805, 1106)
│     ├── 'IMAGES.HER2': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213), (1, 805, 1106)
│     └── 'IMAGES.nuclei': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213), (1, 805, 1106)
├── Labels
│     ├── 'CELLS.main.boundaries.cells': DataTree[yx] (25778, 35416), (12889, 17708), (6444, 8854), (3222, 4427), (1611, 2213), (805, 1106)
│     └── 'CELLS.main.boundaries.nuclei': DataTree[yx] (25778, 35416), (12889, 17708), (6444, 8854), (3222, 4427), (1611, 2213), (805, 1106)
├── Points
│     └── 'TRANSCRIPTS': DataFrame with shape: (42638083, 8) (3D points)
├── Shapes
│     ├── '

#### Naming dialect (short version)

```
{SAMPLE.<uid>..}?<MODALITY>.<locator>[.<locator>...]
```

The `SAMPLE.<uid>..` prefix is added only for an `InSituExperiment` export (see Part 2); a single
`InSituData` produces un-prefixed keys like `IMAGES.CD20`, `CELLS.main.table`,
`CELLS.main.boundaries.cells`, `UNITS.<key>.table`, `TRANSCRIPTS`, `ANNOTATIONS.demo`,
`REGIONS.TMA`. A built `InSituExperiment.build_table()` union table exports as `TABLES.<layer>`
(no `SAMPLE.` prefix even for a multi-sample export - see Part 5).

Current dialect version is **3** (`insitupy._constants.SPATIALDATA_DIALECT_VERSION`), stamped
into `sdata.attrs["insitupy_spatialdata_dialect"]`. The full, versioned spec lives in
`insitupy/spatialdata/DIALECT.md` - treat it as the source of truth if this cell and the code
ever disagree.

In [6]:
sdata.attrs["insitupy_spatialdata_dialect"]

{'version': 3,
 'modalities': ['cells',
  'units',
  'images',
  'transcripts',
  'annotations',
  'regions',
  'tables'],
 'sample_prefix_pattern': 'SAMPLE.<uid>..',
 'slide_id': '0001879',
 'sample_id': 'Replicate 1'}

### 1.3 Round-trip back with `convert_from_spatialdata` (in-memory, no disk)

In [7]:
xd_rt = convert_from_spatialdata(sdata)
xd_rt

2026-07-16 09:30:17 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:30:17 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:30:17 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:30:17 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:30:17 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:30:17 | [INFO] Added 4 new annotations to key 'demo'
2026-07-16 09:30:17 | [INFO] Added 5 new annotations to key 'demo2'
2026-07-16 09:30:17 | [INFO] Added 7 new annotations to key 'demo3'
2026-07-16 09:30:17 | [INFO] Added 3 new regions to key 'demo_regions'
2026-07-16 09:30:17 | [INFO] Added 6 new regions to key 'TMA'


InSituData
Method:		
Slide ID:	0001879
Sample ID:	Replicate 1
UID:		None
Path:		None

    ➤ images
       'CD20':     (25778, 35416)
       'HE':       (25778, 35416, 3)
       'HER2':     (25778, 35416)
       'nuclei':   (25778, 35416)
    ➤ cells
       MultiCellData with main layer 'main'
           table
               AnnData object with n_obs × n_vars = 157600 × 297
               obs: 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_genes_by_counts', 'n_genes', 'leiden'
               var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
               uns: 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'umap'
               obsm: 'X_pca', 'X_umap', 'spatial'
               varm: 'PCs'
               layers: 'counts', 'norm_counts'
               obsp: 'connectivities', 'distances'
           boundaries
               Bound

In [8]:
# Sanity checks: the reconstruction should be a faithful inverse of the export
print("obs_names match:", xd_rt.cells["main"].table.obs_names.equals(xd.cells["main"].table.obs_names))
print("X allclose:      ", np.allclose(xd_rt.cells["main"].table.X.toarray(), xd.cells["main"].table.X.toarray()))
print("is_synced:       ", xd_rt.cells["main"].is_synced)

obs_names match: True
X allclose:       True
is_synced:        True


### 1.4 Persist to zarr and reload with `read_spatialdata`

`read_spatialdata(path)` is a thin convenience wrapper around `spatialdata.read_zarr()` +
`convert_from_spatialdata()`, matching the `insitupy.io` reader convention
(`read_xenium`, `read_visium`, ...).

In [9]:
outpath = CACHE / "test_spatialdata_full.zarr"

In [10]:

sdata.write(outpath, overwrite=True)
print(f"Saved to: {outpath}")

c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\contextlib.py:141: UserWarning: 

Saved to: C:\Users\ge37voy\.cache\InSituPy\test_spatialdata_full.zarr


In [11]:
outpath

WindowsPath('C:/Users/ge37voy/.cache/InSituPy/test_spatialdata_full.zarr')

In [10]:
xd_from_disk = read_spatialdata(outpath)
xd_from_disk

2026-07-16 09:30:20 | [INFO] root_attr: version
2026-07-16 09:30:20 | [INFO] root_attr: multiscales
2026-07-16 09:30:20 | [INFO] root_attr: omero
2026-07-16 09:30:20 | [INFO] datasets [{'path': 's0', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 1.0, 1.0]}, {'type': 'translation', 'translation': [0.0, 0.0, 0.0]}]}, {'path': 's1', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 2.0, 2.0]}, {'type': 'translation', 'translation': [0.0, 0.5, 0.5]}]}, {'path': 's2', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 4.000310366232154, 4.0]}, {'type': 'translation', 'translation': [0.0, 1.5001551831160769, 1.5]}]}, {'path': 's3', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 8.000620732464307, 8.0]}, {'type': 'translation', 'translation': [0.0, 3.5003103662321537, 3.5]}]}, {'path': 's4', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 16.001241464928615, 16.003615002259377]}, {'type': 'translation', 'translation': [0

2026-07-16 09:30:22 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:30:22 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:30:22 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:30:22 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:30:23 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:30:23 | [INFO] Added 4 new annotations to key 'demo'
2026-07-16 09:30:23 | [INFO] Added 5 new annotations to key 'demo2'
2026-07-16 09:30:23 | [INFO] Added 7 new annotations to key 'demo3'
2026-07-16 09:30:23 | [INFO] Added 3 new regions to key 'demo_regions'
2026-07-16 09:30:23 | [INFO] Added 6 new regions to key 'TMA'


InSituData
Method:		
Slide ID:	0001879
Sample ID:	Replicate 1
UID:		None
Path:		None

    ➤ images
       'CD20':     (25778, 35416)
       'HE':       (25778, 35416, 3)
       'HER2':     (25778, 35416)
       'nuclei':   (25778, 35416)
    ➤ cells
       MultiCellData with main layer 'main'
           table
               AnnData object with n_obs × n_vars = 157600 × 297
               obs: 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_genes_by_counts', 'n_genes', 'leiden'
               var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
               uns: 'log1p', 'umap', 'pca', 'leiden', 'neighbors', 'leiden_colors'
               obsm: 'spatial', 'X_umap', 'X_pca'
               varm: 'PCs'
               layers: 'counts', 'norm_counts'
               obsp: 'connectivities', 'distances'
           boundaries
               Bound

In [13]:
xd_from_disk.show()

2026-07-16 07:51:07 | [INFO] Extracting unique gene names from Dask DataFrame...
2026-07-16 07:51:09 | [INFO] Found 541 unique genes


In [14]:
xd_from_disk.path

In [15]:
xd_from_disk.save()

RuntimeError: Cannot save: no project is linked. Use .saveas() to save to a new location.

The reconstructed object has no backing project directory yet - `.saveas(path)` persists it as
a `.insitupy` project before `.save()` can be used.

In [16]:
xd_from_disk.saveas(CACHE / "out/from_spatialdata_native_demo", overwrite=True)
print("Saved.")

2026-07-16 07:56:47 | [INFO] Saving data to C:\Users\ge37voy\.cache\InSituPy\out\from_spatialdata_native_demo


c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\site-packages\dask\array\core.py:3015: PerformanceWarning: The input Dask array will be rechunked along axis 1 with chunk size 4427, but a chunk size divisible by 4096 is required for Dask to write safely to the Zarr array <Array file://C:/Users/ge37voy/.cache/InSituPy/out/from_spatialdata_native_demo/cells/260716-075817606303-674aa2a7/main/boundaries.zarr/masks/cells/3 shape=(3223, 4427) dtype=uint32>. To avoid risk of data loss when writing to this Zarr array, set the "array.chunk-size" configuration parameter to at least the size in bytes of a single on-disk chunk (or shard) of the Zarr array, which in this case is 52805632 bytes. E.g., dask.config.set({"array.chunk-size": 52805632})
  return to_zarr(self, *args, **kwargs)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\site-packages\dask\array\core.py:3015: PerformanceWarning: The input Dask array will be rechunked along axis 1 with chunk size 4427, but a chunk siz

2026-07-16 07:59:49 | [INFO] Saved.
Saved.


In [17]:
xd_from_disk.path

WindowsPath('C:/Users/ge37voy/.cache/InSituPy/out/from_spatialdata_native_demo')

In [ ]:
from insitupy import InSituData

xd_from_disk_rl = InSituData.read(CACHE / "out/from_spatialdata_native_demo")

In [19]:
xd_from_disk_rl.load_all()

In [ ]:
xd_from_disk_rl.show()

2026-07-15 22:51:17 | [INFO] No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


2026-07-15 22:51:31 | [INFO] Extracting unique gene names from Dask DataFrame...
2026-07-15 22:51:32 | [INFO] Found 541 unique genes


C:\Users\ge37voy\Github\insitupy\insitupy\_core\_napari.py:615: FutureWarning: Setting unit on the ScaleBar model is deprecated. Units will instead be computed from the layers in the layerlist. To silence this warning, leave scale_bar unit as `None`, and use `Layer.units` to set units for each layer. Starting in v0.8.0, setting ScaleBar.unit will no longer have an effect. Starting from v0.9.0, it will be removed and raise an exception.
  current_viewer.scale_bar.unit = unit


INFO: New layer '🔬 demo' created.
INFO: New layer '🔬 demo2' created.
INFO: New layer '🔬 demo3' created.
INFO: New layer '🌍 demo_regions' created.
INFO: New layer '🌍 TMA' created.


In [36]:
xd_from_disk.show()

2026-07-15 10:00:51 | [INFO] Extracting unique gene names from Dask DataFrame...
2026-07-15 10:00:53 | [INFO] Found 541 unique genes


## Part 2 - Multi-sample round trip (`InSituExperiment`) + view export

### 2.1 Build a small multi-sample experiment

We reuse the `TMA` regions already present in the demo project's `.annotations` to carve out
several samples via `InSituExperiment.from_regions()`. Transcripts are deleted beforehand purely
to keep region-cropping fast for this demo (see the discussion in
`tutorial_convert_to_spatialdata.ipynb`, section "Create dataset with multiple samples").

In [6]:
# del xd.transcripts

exp = InSituExperiment.from_regions(
    data=xd,
    region_key="TMA",  # column in .annotations containing region IDs
)
exp

C:\Users\ge37voy\AppData\Local\Temp\ipykernel_19436\1493027513.py:3: UserWarning: Transcript data will be loaded into memory to speed up region cropping. This may require substantial RAM for large datasets.
  exp = InSituExperiment.from_regions(


2026-07-21 13:20:25 | [INFO] Loading transcripts into memory...
2026-07-21 13:20:31 | [INFO] Transcripts loaded: 42,638,083 rows.


Iterating regions: 100%|██████████| 6/6 [00:39<00:00,  6.56s/it]


InSituExperiment
Path:		None
    ➤ data
        6 samples
        3 metadata columns:
        "uid", "region_key", "region_name"
        Loaded modalities
            cells: 6/6
            images: 6/6
            transcripts: 6/6
            annotations: 3/6
            regions: 6/6
    ➤ filters
        Base filters: none
        
        Composite filters: none
    ➤ table
        no tables built

### 2.2 Export/import the full experiment

In [7]:
sdexp = convert_to_spatialdata(exp)
sdexp

2026-07-21 13:21:21 | [INFO] No case-insensitive conflicts found.


SpatialData object
├── Images
│     ├── 'SAMPLE.2e90405c..IMAGES.CD20': DataTree[cyx] (1, 4706, 4706), (1, 2353, 2353), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.2e90405c..IMAGES.HE': DataTree[cyx] (3, 4706, 4706), (3, 2353, 2353), (3, 1176, 1176), (3, 588, 588), (3, 294, 294), (3, 147, 147)
│     ├── 'SAMPLE.2e90405c..IMAGES.HER2': DataTree[cyx] (1, 4706, 4706), (1, 2353, 2353), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.2e90405c..IMAGES.nuclei': DataTree[cyx] (1, 4706, 4706), (1, 2353, 2353), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.3c727641..IMAGES.CD20': DataTree[cyx] (1, 4706, 4705), (1, 2353, 2352), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.3c727641..IMAGES.HE': DataTree[cyx] (3, 4706, 4705), (3, 2353, 2352), (3, 1176, 1176), (3, 588, 588), (3, 294, 294), (3, 147, 147)
│     ├── 'SAMPLE.3c727641..IMAGES.HER2': DataTree[cyx] (1, 4706, 

In [13]:
exp_rt = convert_from_spatialdata(sdexp)
print(type(exp_rt).__name__, "with", len(exp_rt), "samples")
print("uids:", sorted(exp_rt.metadata["uid"]))

2026-07-16 09:31:01 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:31:01 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:31:01 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:31:01 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:31:01 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:31:01 | [INFO] Added 1 new regions to key 'TMA'
2026-07-16 09:31:01 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:31:01 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:31:01 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:31:01 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:31:01 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:31:01 | [INFO] Added 1 new regions to key 'TMA'
2026-07-16 09:31:01 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:31:01 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:31:01 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:31:01 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:31:01 | [INFO] Extracted pixel size 0.2125
2026-07-16 09:31:01 |

In [14]:
exp_rt.show(1)

2026-07-16 09:31:06 | [INFO] Extracting unique gene names from Dask DataFrame...
2026-07-16 09:31:06 | [INFO] Found 541 unique genes


### 2.3 View dispatch - export a lazy subset view directly

`experiment[i:j]` always returns an `InSituExperimentView` sharing its datasets with the parent
(no copy). Before WP2+WP3, exporting a view either crashed or silently dropped the concatenated
table when one had been built (`_is_experiment()` only recognized a bare `InSituExperiment`); it
is now dispatched correctly, same as a full experiment.

In [15]:
view = exp[0:2]
print(type(view).__name__, "covering", len(view), "samples")

sdata_view = convert_to_spatialdata(view)
sdata_view

InSituExperimentView covering 2 samples
2026-07-16 09:46:26 | [INFO] No case-insensitive conflicts found.


SpatialData object
├── Images
│     ├── 'SAMPLE.78299059..IMAGES.CD20': DataTree[cyx] (1, 4706, 4705), (1, 2353, 2352), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.78299059..IMAGES.HE': DataTree[cyx] (3, 4706, 4705), (3, 2353, 2352), (3, 1176, 1176), (3, 588, 588), (3, 294, 294), (3, 147, 147)
│     ├── 'SAMPLE.78299059..IMAGES.HER2': DataTree[cyx] (1, 4706, 4705), (1, 2353, 2352), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.78299059..IMAGES.nuclei': DataTree[cyx] (1, 4706, 4705), (1, 2353, 2352), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.f66b45c1..IMAGES.CD20': DataTree[cyx] (1, 4706, 4706), (1, 2353, 2353), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.f66b45c1..IMAGES.HE': DataTree[cyx] (3, 4706, 4706), (3, 2353, 2353), (3, 1176, 1176), (3, 588, 588), (3, 294, 294), (3, 147, 147)
│     ├── 'SAMPLE.f66b45c1..IMAGES.HER2': DataTree[cyx] (1, 4706, 

In [16]:
sdata_view

SpatialData object
├── Images
│     ├── 'SAMPLE.78299059..IMAGES.CD20': DataTree[cyx] (1, 4706, 4705), (1, 2353, 2352), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.78299059..IMAGES.HE': DataTree[cyx] (3, 4706, 4705), (3, 2353, 2352), (3, 1176, 1176), (3, 588, 588), (3, 294, 294), (3, 147, 147)
│     ├── 'SAMPLE.78299059..IMAGES.HER2': DataTree[cyx] (1, 4706, 4705), (1, 2353, 2352), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.78299059..IMAGES.nuclei': DataTree[cyx] (1, 4706, 4705), (1, 2353, 2352), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.f66b45c1..IMAGES.CD20': DataTree[cyx] (1, 4706, 4706), (1, 2353, 2353), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.f66b45c1..IMAGES.HE': DataTree[cyx] (3, 4706, 4706), (3, 2353, 2353), (3, 1176, 1176), (3, 588, 588), (3, 294, 294), (3, 147, 147)
│     ├── 'SAMPLE.f66b45c1..IMAGES.HER2': DataTree[cyx] (1, 4706, 

## Part 3 - Spatial units modality (Visium)

Spatial units (Visium spots, or any other polygon-based unit layer added via
`InSituData.add_units()`) export as a `TableModel` + `ShapesModel` pair - the same
`region`/`region_key`/`instance_key` linkage as cells, but using the units' own real polygon
geometries instead of synthesized circles. `visium_human_breast_cancer()` downloads a small,
self-contained Visium dataset for this section (images are dropped to keep it focused on
`units`).

### 3.1 Load the Visium demo dataset (units-only, no cells)

In [19]:
from insitupy.datasets import visium_human_breast_cancer

visium = visium_human_breast_cancer()
# del visium.images  # keep this section focused on the units modality
visium

2026-07-16 09:47:21 | [INFO] H5 file exists. Checking md5sum...
2026-07-16 09:47:21 | [INFO] The h5 file md5sum matches. Download is skipped. To force download set `overwrite=True`.
2026-07-16 09:47:21 | [INFO] Spatial directory exists. Download is skipped. To force download set `overwrite=True`.
2026-07-16 09:47:21 | [INFO] Visium data structure is ready at C:\Users\ge37voy\.cache\InSituPy\demo_datasets\visium_hbreastcancer\CytAssist_FFPE_Human_Breast_Cancer
2026-07-16 09:47:21 | [INFO] Dataset contains:
2026-07-16 09:47:21 | [INFO] - filtered_feature_bc_matrix.h5
2026-07-16 09:47:21 | [INFO] - spatial/ directory
2026-07-16 09:47:21 | [INFO] Reading Visium data with spatialdata-io...


c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\site-packages\anndata\_core\anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\site-packages\spatialdata\models\models.py:1267: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)


2026-07-16 09:47:23 | [INFO] Adding images...
2026-07-16 09:47:23 | [INFO] Adding spatial units...
2026-07-16 09:47:23 | [INFO] Converting 4992 Point geometries with radius to circular polygons using buffer.
2026-07-16 09:47:23 | [WARNING] Indices in `.shapes` do not match `.data.obs_names`. Shapes will be renamed according to the `obs_names`. For this to be valid, please make sure that the order of elements in `.shapes` and `.data` matches.


InSituData
Method:		visium
Slide ID:	None
Sample ID:	None
UID:		None
Path:		None

    ➤ images
       'hires':    (2000, 1809, 3)
       'lowres':   (600, 543, 3)
    ➤ units
       MultiSpatialUnitsData with main layer 'visium'
           SpatialUnitsData (Type: 'visium')
               .table: 4992 obs x 18085 vars
               .shapes: 4992 geometries

### 3.2 Export units to SpatialData

In [20]:
sdata_units = convert_to_spatialdata(visium)
sdata_units

2026-07-16 09:47:25 | [INFO] No case-insensitive conflicts found.


SpatialData object
├── Images
│     ├── 'IMAGES.hires': DataTree[cyx] (3, 2000, 1809), (3, 1000, 904), (3, 500, 452), (3, 250, 226), (3, 125, 113), (3, 62, 56)
│     └── 'IMAGES.lowres': DataTree[cyx] (3, 600, 543), (3, 300, 271), (3, 150, 135), (3, 75, 67), (3, 37, 33), (3, 18, 16)
├── Shapes
│     └── 'UNITS.visium.shapes': GeoDataFrame shape: (4992, 2) (2D shapes)
└── Tables
      └── 'UNITS.visium.table': AnnData (4992, 18085)
with coordinate systems:
    ▸ 'global', with elements:
        IMAGES.hires (Images), IMAGES.lowres (Images)
    ▸ 'visium', with elements:
        UNITS.visium.shapes (Shapes)
    ▸ 'visium_downscaled_hires', with elements:
        UNITS.visium.shapes (Shapes)
    ▸ 'visium_downscaled_lowres', with elements:
        UNITS.visium.shapes (Shapes)

### 3.3 Round-trip units back through the dialect reader

In [21]:
visium_rt = convert_from_spatialdata(sdata_units)
visium_rt

2026-07-16 09:47:28 | [INFO] Extracted pixel size 5.906139790390711
2026-07-16 09:47:28 | [INFO] Extracted pixel size 19.687132776192747


InSituData
Method:		
Slide ID:	None
Sample ID:	None
UID:		None
Path:		None

    ➤ images
       'hires':    (2000, 1809, 3)
       'lowres':   (600, 543, 3)
    ➤ units
       MultiSpatialUnitsData with main layer 'visium'
           SpatialUnitsData (Type: 'visium')
               .table: 4992 obs x 18085 vars
               .shapes: 4992 geometries

In [22]:
print("original: ", visium.units["visium"].table.shape)
print("roundtrip:", visium_rt.units["visium"].table.shape)

original:  (4992, 18085)
roundtrip: (4992, 18085)


## Part 4 - Foreign-store import with `convert_from_foreign_spatialdata`

`convert_from_spatialdata` (Parts 1-3) only reads stores InSituPy itself wrote - it requires the
`insitupy_spatialdata_dialect` descriptor in `sdata.attrs`. `convert_from_foreign_spatialdata` is
the counterpart for everything else: raw `spatialdata_io` reader output, or any other
labels-native store following the standard SpatialData `TableModel` annotation contract
(`region`/`region_key`/`instance_key` in `table.uns["spatialdata_attrs"]`).

As of the keyed-dict refactor, `images`, `cells`, and `units` are each `{name: spec}` dicts - one
entry per InSituData-side image/cell-layer/units-layer - because `InSituData` supports multiple
cell layers (`MultiCellData`) and multiple spatial-units layers (`MultiSpatialUnitsData`), each
built from its own SpatialData table. The first entry of each dict becomes the main layer.
`transcripts` stays a scalar SpatialData points key, since `InSituData.transcripts` is
single-cardinality. Every modality parameter defaults to `None` - nothing is imported unless
asked.

### 4.1 Xenium-style import with explicit specs

This mirrors what `read_xenium(path, backend="spatialdata")` does internally
(`insitupy/io/data.py`). We force `cells_as_circles=True` so the store has a `cell_circles`
shapes element to point `cells_key` at explicitly.

In [23]:
from spatialdata_io import xenium as sdio_xenium

datapath = CACHE / "demo_datasets/xenium_hbreastcancer/output-XETG00000__slide_id__hbreastcancer"
pixel_size = 0.2125  # Xenium, um/pixel

sdata_raw = sdio_xenium(datapath, cells_as_circles=True)
sdata_raw

SpatialData object
├── Images
│     ├── 'morphology_focus': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213)
│     └── 'morphology_mip': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (25778, 35416), (12889, 17708), (6444, 8854), (3222, 4427), (1611, 2213)
│     └── 'nucleus_labels': DataTree[yx] (25778, 35416), (12889, 17708), (6444, 8854), (3222, 4427), (1611, 2213)
├── Points
│     └── 'transcripts': DataFrame with shape: (8000000, 8) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (167780, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (167780, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (167780, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (167780, 313)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), morphology_mip (

In [24]:
xd_foreign = convert_from_foreign_spatialdata(
    sdata=sdata_raw,
    images={
        "nuclei": {"key": "morphology_focus", "pixel_size": pixel_size},
        "mip":    {"key": "morphology_mip",   "pixel_size": pixel_size},
    },
    cells={"main": {
        "table_key": "table",
        "cells_key": "cell_circles",
        "cell_boundaries_data": ("cell_labels", pixel_size),
        "nucleus_boundaries_data": ("nucleus_labels", pixel_size),
    }},
    transcripts="transcripts",
    slide_id="slide_demo",
    sample_id="sample_demo",
    method_name="Xenium",
)
xd_foreign

2026-07-16 09:48:05 | [INFO] Adding images...
2026-07-16 09:48:05 | [INFO] Adding cell data...
2026-07-16 09:48:05 | [WARNING] Spatial coordinates in `obsm['spatial']` are overwritten using centroids from `'cell_circles'`.
2026-07-16 09:48:05 | [INFO] Adding transcripts...


InSituData
Method:		Xenium
Slide ID:	slide_demo
Sample ID:	sample_demo
UID:		None
Path:		None

    ➤ images
       'nuclei':   (25778, 35416)
       'mip':      (25778, 35416)
    ➤ cells
       MultiCellData with main layer 'main'
           table
               AnnData object with n_obs × n_vars = 167780 × 313
               obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region'
               var: 'gene_ids', 'feature_types', 'genome'
               uns: 'spatialdata_attrs'
               obsm: 'spatial'
           boundaries
               BoundariesData object with 2 entries:
                   cells
                   nuclei
    ➤ transcripts
       DataFrame with shape <dask_expr.expr.Scalar: expr=(RenameFrame(frame=Assign(frame=Assign(frame=Assign(frame=Assign(frame=Assign(frame=ColumnsSetter(frame=Assign(frame=ReadParquetFSSpec(4cda62d))[['x_location', 'y_location', 'z_location']], columns=[

### 4.2 Equivalent convenience wrapper: `read_xenium(..., backend="spatialdata")`

In [25]:
from insitupy.io import read_xenium

xd_direct = read_xenium(datapath, backend="spatialdata")
xd_direct

2026-07-16 09:48:10 | [INFO] Reading Xenium data with spatialdata-io backend...
2026-07-16 09:48:18 | [INFO] Adding images...
2026-07-16 09:48:18 | [INFO] Adding cell data...
2026-07-16 09:48:18 | [WARNING] Spatial coordinates in `obsm['spatial']` are overwritten using centroids derived from 'cell_labels'.
2026-07-16 09:48:54 | [INFO] Adding transcripts...


InSituData
Method:		Xenium
Slide ID:	None
Sample ID:	None
UID:		None
Path:		None

    ➤ images
       'nuclei':   (25778, 35416)
       'mip':      (25778, 35416)
    ➤ cells
       MultiCellData with main layer 'main'
           table
               AnnData object with n_obs × n_vars = 167780 × 313
               obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region'
               var: 'gene_ids', 'feature_types', 'genome'
               uns: 'spatialdata_attrs'
               obsm: 'spatial'
           boundaries
               BoundariesData object with 2 entries:
                   cells
                   nuclei
    ➤ transcripts
       DataFrame with shape <dask_expr.expr.Scalar: expr=(RenameFrame(frame=Assign(frame=Assign(frame=Assign(frame=Assign(frame=Assign(frame=ColumnsSetter(frame=Assign(frame=ReadParquetFSSpec(4cda62d))[['x_location', 'y_location', 'z_location']], columns=['x', 'y', 'z'

### 4.3 Visium-style units-only import (mirrors `read_visium` internals)

`read_visium()` (`insitupy/io/data.py`) reads `spatial/scalefactors_json.json` to convert
`spatialdata_io`'s pixel-unit hires/lowres image and spot geometries into physical (micron)
units, then calls `convert_from_foreign_spatialdata(..., units={...}, cells=None)` - no `.cells`
layer, since Visium spots are spatial units, not segmented cells.

In [ ]:
import json

from shapely import affinity
from spatialdata_io import visium as sdio_visium

data_dir = CACHE / "demo_datasets/visium_hbreastcancer/CytAssist_FFPE_Human_Breast_Cancer"
dataset_id = "visium"
fullres_pixel_size = 0.5476  # um/pixel, known resolution of this dataset (see visium_human_breast_cancer())

sf_file = data_dir / "spatial" / "scalefactors_json.json"
scale_factors = json.loads(sf_file.read_text())
hires_pixel_size = fullres_pixel_size / scale_factors["tissue_hires_scalef"]
lowres_pixel_size = fullres_pixel_size / scale_factors["tissue_lowres_scalef"]

sdata_visium_raw = sdio_visium(path=data_dir, dataset_id=dataset_id)

# spatialdata_io reports spot geometries in pixel units; rescale to microns (same as read_visium())
sdata_visium_raw[dataset_id]["geometry"] = sdata_visium_raw[dataset_id]["geometry"].apply(
    lambda geom: affinity.scale(geom, xfact=fullres_pixel_size, yfact=fullres_pixel_size, origin=(0, 0))
)
if "radius" in sdata_visium_raw[dataset_id].columns:
    sdata_visium_raw[dataset_id]["radius"] = sdata_visium_raw[dataset_id]["radius"] * fullres_pixel_size

sdata_visium_raw

c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\site-packages\anndata\_core\anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\site-packages\spatialdata\models\models.py:1267: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)


SpatialData object
├── Images
│     ├── 'visium_hires_image': DataArray[cyx] (3, 2000, 1809)
│     └── 'visium_lowres_image': DataArray[cyx] (3, 600, 543)
├── Shapes
│     └── 'visium': GeoDataFrame shape: (4992, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (4992, 18085)
with coordinate systems:
    ▸ 'visium', with elements:
        visium_hires_image (Images), visium_lowres_image (Images), visium (Shapes)
    ▸ 'visium_downscaled_hires', with elements:
        visium_hires_image (Images), visium (Shapes)
    ▸ 'visium_downscaled_lowres', with elements:
        visium_lowres_image (Images), visium (Shapes)

In [27]:
xd_visium_foreign = convert_from_foreign_spatialdata(
    sdata=sdata_visium_raw,
    images={
        "hires":  {"key": f"{dataset_id}_hires_image",  "pixel_size": hires_pixel_size,  "is_rgb": True},
        "lowres": {"key": f"{dataset_id}_lowres_image", "pixel_size": lowres_pixel_size, "is_rgb": True},
    },
    units={"visium": {"table_key": "table", "units_key": dataset_id, "unit_type": "visium"}},
    transcripts=None,  # Visium has no single-molecule transcripts
    slide_id="visium_demo",
    sample_id="sample_demo",
    method_name="visium",
)
xd_visium_foreign

2026-07-16 10:04:24 | [INFO] Adding images...
2026-07-16 10:04:24 | [INFO] Adding spatial units...
2026-07-16 10:04:25 | [INFO] Converting 4992 Point geometries with radius to circular polygons using buffer.
2026-07-16 10:04:25 | [WARNING] Indices in `.shapes` do not match `.data.obs_names`. Shapes will be renamed according to the `obs_names`. For this to be valid, please make sure that the order of elements in `.shapes` and `.data` matches.


InSituData
Method:		visium
Slide ID:	visium_demo
Sample ID:	sample_demo
UID:		None
Path:		None

    ➤ images
       'hires':    (2000, 1809, 3)
       'lowres':   (600, 543, 3)
    ➤ units
       MultiSpatialUnitsData with main layer 'visium'
           SpatialUnitsData (Type: 'visium')
               .table: 4992 obs x 18085 vars
               .shapes: 4992 geometries

### 4.4 Minimal labels-native import - full auto-detection chain

Calling `spatialdata_io.xenium()` with its current default (`cells_as_circles=False`) produces a
purely labels-native store: no `cell_circles` shapes element, table `region` points straight at
the `cell_labels` labels element. `cells={"main": {"table_key": "table"}}` is then enough -
`cells_key`/`cell_boundaries_data`/pixel size are auto-detected from the table's own
`spatialdata_attrs`, segmentation identity (`seg_mask_value`) comes from the table's real
`instance_key` column, and cell centroids are derived from label-mask regionprops since there is
no shapes element to read them from.

In [28]:
sdata_labels_native = sdio_xenium(datapath)  # cells_as_circles=False by default
sdata_labels_native

SpatialData object
├── Images
│     ├── 'morphology_focus': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213)
│     └── 'morphology_mip': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (25778, 35416), (12889, 17708), (6444, 8854), (3222, 4427), (1611, 2213)
│     └── 'nucleus_labels': DataTree[yx] (25778, 35416), (12889, 17708), (6444, 8854), (3222, 4427), (1611, 2213)
├── Points
│     └── 'transcripts': DataFrame with shape: (8000000, 8) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (167780, 1) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (167780, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (167780, 313)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), morphology_mip (Images), cell_labels (Labels), nucleus_labels (Labels), transcripts (P

In [29]:
xd_auto = convert_from_foreign_spatialdata(
    sdata_labels_native,
    cells={"main": {"table_key": "table"}},
)
xd_auto

2026-07-16 10:13:01 | [INFO] Adding cell data...
2026-07-16 10:13:01 | [INFO] Extracted pixel size 1.0
2026-07-16 10:13:01 | [WARNING] Spatial coordinates in `obsm['spatial']` are overwritten using centroids derived from 'cell_labels'.


InSituData
Method:		
Slide ID:	None
Sample ID:	None
UID:		None
Path:		None

    ➤ cells
       MultiCellData with main layer 'main'
           table
               AnnData object with n_obs × n_vars = 167780 × 313
               obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region'
               var: 'gene_ids', 'feature_types', 'genome'
               uns: 'spatialdata_attrs'
               obsm: 'spatial'
           boundaries
               BoundariesData object with 2 entries:
                   cells

In [32]:
print("boundaries pixel size (auto-detected):", xd_auto.cells["main"].boundaries.metadata["cells"]["pixel_size"])
print("is_synced:", xd_auto.cells["main"].is_synced)

boundaries pixel size (auto-detected): 1.0
is_synced: True


In [33]:
xd_auto

InSituData
Method:		
Slide ID:	None
Sample ID:	None
UID:		None
Path:		None

    ➤ cells
       MultiCellData with main layer 'main'
           table
               AnnData object with n_obs × n_vars = 167780 × 313
               obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region'
               var: 'gene_ids', 'feature_types', 'genome'
               uns: 'spatialdata_attrs'
               obsm: 'spatial'
           boundaries
               BoundariesData object with 2 entries:
                   cells

### 4.5 Multi-layer cells import - two independent tables, two named cell layers

This is the exact capability the old shared `table_key` param could not express: two cell layers
sourced from two *distinct* SpatialData tables, landing under their own given names
(`"main"`/`"secondary"`), with independent pixel sizes. We build a tiny synthetic SpatialData
object by hand (mirroring `tests/test_spatialdata_foreign_import.py::TestMultiLayerImport`) since
no real downloadable dataset ships two independent segmentation tables.

In [34]:
import pandas as pd
from anndata import AnnData
from spatialdata import SpatialData
from spatialdata.models import Labels2DModel, TableModel
from spatialdata.transformations import Identity, Scale
from xarray import DataArray

n_cells = 3
size = n_cells * 4

# --- layer "main": labels at scale 1.0 ---
ids_a = np.arange(1, n_cells + 1)
mask_a = np.zeros((size, size), dtype=np.uint32)
for i, value in enumerate(ids_a):
    mask_a[i * 4, i * 4] = value
labels_a = Labels2DModel.parse(DataArray(mask_a, dims=("y", "x")), transformations={"global": Identity()})
obs_a = pd.DataFrame({"cell_id": ids_a, "region": pd.Categorical(["labels_a"] * n_cells)})
adata_a = AnnData(
    X=np.random.default_rng(0).random((n_cells, 2)),
    obs=obs_a, var=pd.DataFrame(index=["gene_0", "gene_1"]),
)
table_a = TableModel.parse(adata_a, region="labels_a", region_key="region", instance_key="cell_id")

# --- layer "secondary": labels at scale 2.0, non-overlapping id range ---
ids_b = np.arange(101, 101 + n_cells)
mask_b = np.zeros((size, size), dtype=np.uint32)
for i, value in enumerate(ids_b):
    mask_b[i * 4, i * 4] = value
labels_b = Labels2DModel.parse(
    DataArray(mask_b, dims=("y", "x")), transformations={"global": Scale([2.0, 2.0], axes=("x", "y"))},
)
obs_b = pd.DataFrame({"cell_id": ids_b, "region": pd.Categorical(["labels_b"] * n_cells)})
adata_b = AnnData(
    X=np.random.default_rng(1).random((n_cells, 2)),
    obs=obs_b, var=pd.DataFrame(index=["gene_0", "gene_1"]),
)
table_b = TableModel.parse(adata_b, region="labels_b", region_key="region", instance_key="cell_id")

sdata_multi = SpatialData(
    labels={"labels_a": labels_a, "labels_b": labels_b},
    tables={"table_a": table_a, "table_b": table_b},
)
sdata_multi

c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


SpatialData object
├── Labels
│     ├── 'labels_a': DataArray[yx] (12, 12)
│     └── 'labels_b': DataArray[yx] (12, 12)
└── Tables
      ├── 'table_a': AnnData (3, 2)
      └── 'table_b': AnnData (3, 2)
with coordinate systems:
    ▸ 'global', with elements:
        labels_a (Labels), labels_b (Labels)

In [35]:
xd_multi = convert_from_foreign_spatialdata(
    sdata_multi,
    cells={
        # "main": explicit boundaries key, pixel size 1.0
        "main": {"table_key": "table_a", "cell_boundaries_data": ("labels_a", 1.0)},
        # "secondary": nothing explicit - cells_key/cell_boundaries_data/pixel size all
        # auto-detected from table_b's own region, independently of "main"'s explicit values
        "secondary": {"table_key": "table_b"},
    },
)

print("layers:  ", list(xd_multi.cells.keys()))
print("main key:", xd_multi.cells.main_key)
print("main pixel size:     ", xd_multi.cells["main"].boundaries.metadata["cells"]["pixel_size"])
print("secondary pixel size:", xd_multi.cells["secondary"].boundaries.metadata["cells"]["pixel_size"])

2026-07-16 10:17:03 | [INFO] Adding cell data...
2026-07-16 10:17:03 | [INFO] Extracted pixel size 2.0
layers:   ['main', 'secondary']
main key: main
main pixel size:      1.0
secondary pixel size: 2.0


In [38]:
xd_multi

InSituData
Method:		
Slide ID:	None
Sample ID:	None
UID:		None
Path:		None

    ➤ cells
       MultiCellData with main layer 'main'
           table
               AnnData object with n_obs × n_vars = 3 × 2
               obs: 'cell_id', 'region'
               uns: 'spatialdata_attrs'
               obsm: 'spatial'
           boundaries
               BoundariesData object with 2 entries:
                   cells
       
       Additional layers with keys: 'secondary'

In [37]:
xd_multi.show()

C:\Users\ge37voy\Github\insitupy\insitupy\_core\_napari.py:564: UserWarning: No images found.
  warn("No images found.")


### 4.6 Spec validation guardrails

A generic validator (`_validate_foreign_spec`) rejects non-dict specs, missing required keys, and
unknown keys (typo protection) uniformly across `images`/`cells`/`units`.

In [39]:
try:
    convert_from_foreign_spatialdata(sdata_multi, cells={"main": {}})
except ValueError as e:
    print("Missing required key:", e)

2026-07-16 13:35:06 | [INFO] Adding cell data...
Missing required key: cells spec for 'main' is missing required key(s): ['table_key']. Allowed keys: ['table_key', 'cells_key', 'cell_boundaries_data', 'nucleus_boundaries_data']


In [40]:
try:
    convert_from_foreign_spatialdata(
        sdata_multi, cells={"main": {"table_key": "table_a", "tabel_key": "table_a"}},  # typo
    )
except ValueError as e:
    print("Unknown key (typo guard):", e)

2026-07-16 13:35:14 | [INFO] Adding cell data...
Unknown key (typo guard): cells spec for 'main' has unknown key(s): ['tabel_key']. Allowed keys: ['table_key', 'cells_key', 'cell_boundaries_data', 'nucleus_boundaries_data']


In [41]:
try:
    convert_from_foreign_spatialdata(sdata_multi, units={"visium": {"table_key": "table_a"}})  # missing units_key
except ValueError as e:
    print("Missing units_key:", e)

2026-07-16 13:35:24 | [INFO] Adding spatial units...
Missing units_key: units spec for 'visium' is missing required key(s): ['units_key']. Allowed keys: ['table_key', 'units_key', 'unit_type']


## Part 5 - Concatenated table export/import (`TABLES.<layer>`)

`InSituExperiment.build_table()` writes an on-disk, per-cells-layer concatenated union `AnnData`
(outer join across samples) plus a gene-presence record letting readers reconstruct the correct
*inner* gene set on demand. `convert_to_spatialdata(..., include_concat_tables=True)` (the
default) carries this element into the SpatialData export as `TABLES.<layer>` - experiment-level,
no `SAMPLE.` prefix. `convert_table_from_spatialdata()` reapplies the same inner-over-covered
reconstruction to read it back, either for the full experiment or a label-subset (view).

### 5.1 Save the experiment and build a concatenated table

`build_table()` writes to `{experiment_path}/tables/{cells_layer}.zarr`, so the experiment needs a
backing path first.

In [42]:
exp_path = CACHE / "out/spatialdata_demo_experiment"
exp.saveas(exp_path, overwrite=True)
exp.build_table()
exp.table["main"]

100%|██████████| 6/6 [01:19<00:00, 13.32s/it]


Collected warnings:
  [PerformanceWarning] The input Dask array will be rechunked along axis 0 with chunk size 4706, but a chunk size divisible by 4096 is required for Dask to write safely to the Zarr array <Array file://C:/Users/ge37voy/.cache/InSituPy/out/spatialdata_demo_experiment.__ispy_tmp__/data-000/cells/260716-134909598148-0325efc1/main/boundaries.zarr/masks/cells/0 shape=(4706, 4706) dtype=uint32>. To avoid risk of data loss when writing to this Zarr array, set the "array.chunk-size" configuration parameter to at least the size in bytes of a single on-disk chunk (or shard) of the Zarr array, which in this case is 67108864 bytes. E.g., dask.config.set({"array.chunk-size": 67108864})
  [PerformanceWarning] The input Dask array will be rechunked along axis 0 with chunk size 4706, but a chunk size divisible by 4096 is required for Dask to write safely to the Zarr array <Array file://C:/Users/ge37voy/.cache/InSituPy/out/spatialdata_demo_experiment.__ispy_tmp__/data-000/cells/26071

... storing 'region_key' as categorical
... storing 'region_name' as categorical
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\site-packages\anndata\_io\zarr.py:44: UserWarning: Writing zarr v2 data will no longer be the default in the next minor release. v3 data will be written by default. If you are explicitly setting this configuration, consider migrating to the zarr v3 file format.
  f = open_write_group(store)


2026-07-16 13:50:23 | [INFO] Built concatenated table at 'C:\Users\ge37voy\.cache\InSituPy\out\spatialdata_demo_experiment\tables\main.zarr' (17820 cells, 297 genes in union).


View of AnnData object with n_obs × n_vars = 17820 × 297
    obs: 'uid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_genes_by_counts', 'n_genes', 'leiden', 'region_key', 'region_name'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: '_insitupy_build_params', '_insitupy_gene_presence', '_insitupy_presence_labels', '_insitupy_table_format_version'
    obsm: 'spatial'

### 5.2 Export with the concatenated table included

In [43]:
sdata_exp_table = convert_to_spatialdata(exp)  # include_concat_tables=True by default
print("TABLES.main present:", "TABLES.main" in sdata_exp_table.tables)
sdata_exp_table.tables["TABLES.main"]

2026-07-16 13:52:14 | [INFO] No case-insensitive conflicts found.
TABLES.main present: True


AnnData object with n_obs × n_vars = 17820 × 297
    obs: 'uid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_genes_by_counts', 'n_genes', 'leiden', 'region_key', 'region_name', 'cell_id', 'region'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: '_insitupy_build_params', '_insitupy_gene_presence', '_insitupy_presence_labels', '_insitupy_table_format_version', 'spatialdata_attrs'
    obsm: 'spatial'

### 5.3 Reconstruct the full-experiment table from the store

In [47]:
exp.table["main"].X

dask.array<getitem, shape=(17820, 297), dtype=float32, chunksize=(1000, 297), chunktype=scipy.csr_matrix>

In [54]:
exp.table["main"].X.compute().toarray()

array([[0.        , 0.        , 1.6030685 , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 2.176171  , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 2.7006152 , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 1.2456657 , ..., 0.        , 0.        ,
        0.        ],
       [1.1373982 , 0.        , 0.        , ..., 0.        , 0.        ,
        0.95126516],
       [0.        , 0.61615205, 0.82315874, ..., 0.        , 0.        ,
        1.4832294 ]], shape=(17820, 297), dtype=float32)

In [55]:
recon_full = convert_table_from_spatialdata(sdata_exp_table, "main")

print("obs_names match:", recon_full.obs_names.equals(exp.table["main"].obs_names))
print("X allclose:     ", np.allclose(recon_full.X.toarray(), exp.table["main"].X.compute().toarray()))
recon_full

obs_names match: True
X allclose:      True


View of AnnData object with n_obs × n_vars = 17820 × 297
    obs: 'uid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_genes_by_counts', 'n_genes', 'leiden', 'region_key', 'region_name', 'cell_id', 'region'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: '_insitupy_build_params', '_insitupy_gene_presence', '_insitupy_presence_labels', '_insitupy_table_format_version', 'spatialdata_attrs'
    obsm: 'spatial'

### 5.4 Reconstruct a view-scoped (subset) table

`covered_labels` reconstructs the inner-over-that-subset, row-filtered table - equivalent to
`view.table[layer]`, which recovers genes shared by the subset but absent from the full
experiment.

In [57]:
view2 = exp[0:1]
uid0 = view2.metadata["uid"].iloc[0]

recon_view = convert_table_from_spatialdata(sdata_exp_table, "main", covered_labels=[uid0])

print("var_names match:", recon_view.var_names.equals(view2.table["main"].var_names))
print("X allclose:     ", np.allclose(recon_view.X.toarray(), view2.table["main"].X.compute().toarray()))
recon_view

2026-07-16 16:26:26 | [WARNING] You are accessing a copy of the metadata. Changes to this DataFrame will not affect the internal metadata. Use `add_metadata_column()` or `append_metadata()` to add new information to the metadata.
var_names match: True
X allclose:      True


View of AnnData object with n_obs × n_vars = 4878 × 297
    obs: 'uid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_genes_by_counts', 'n_genes', 'leiden', 'region_key', 'region_name', 'cell_id', 'region'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: '_insitupy_build_params', '_insitupy_gene_presence', '_insitupy_presence_labels', '_insitupy_table_format_version', 'spatialdata_attrs'
    obsm: 'spatial'

## Summary

| Function | Section(s) | What it covers |
|---|---|---|
| `convert_to_spatialdata` | 1.2, 2.2, 2.3, 3.2, 5.2 | `InSituData` / `InSituExperiment` / `InSituExperimentView` -> `SpatialData` (images, cells, units, transcripts, annotations, regions, concatenated tables) |
| `convert_from_spatialdata` | 1.3, 2.2, 3.3 | Dialect-driven inverse of the above - single sample and multi-sample |
| `read_spatialdata` | 1.4 | `spatialdata.read_zarr()` + `convert_from_spatialdata()` in one call |
| `convert_from_foreign_spatialdata` | 4.1-4.6 | Keyed-dict importer for non-InSituPy stores - images/cells/units specs, multi-layer import, auto-detection, spec validation |
| `convert_table_from_spatialdata` | 5.3, 5.4 | Reconstructs `build_table()`'s concatenated union table from a `TABLES.<layer>` element, full-experiment or view-scoped |

See `insitupy/spatialdata/DIALECT.md` for the full, versioned naming spec, and
`tests/test_spatialdata_*.py` for the underlying automated coverage this notebook mirrors by
hand.